# HW01-B — SQL, Latency, and Metabase

The business team does not care that your notebook works. They want a dashboard that opens fast.

Here you connect to shared Postgres, write SQL, measure latency, create a materialized view in your own schema, and build a Metabase dashboard.

## Submission discipline

This is individual work.

Work locally. Push to GitHub. Use the shared server services through URLs and credentials. Do not SSH into the server.

Do not commit `.env`, `.venv/`, passwords.

## Credentials and shared services

Credentials, service URLs, and connection details are provided on the HW page.

Use those exact values. Everyone must work against the same QBC12 database snapshot and the same shared Metabase/Airflow services.

Do not paste credentials into notebook markdown. Do not commit `.env` files. Do not screenshot passwords.


## Useful references

- PostgreSQL `EXPLAIN`: https://www.postgresql.org/docs/current/sql-explain.html
- PostgreSQL using `EXPLAIN`: https://www.postgresql.org/docs/current/using-explain.html
- Metabase questions: https://www.metabase.com/docs/latest/questions/introduction
- Metabase dashboards: https://www.metabase.com/docs/latest/dashboards/introduction

if you cannot open any one of these contact me : Bale (arianaghamohseni, image of a scared chicken), or Telegram (@arianaghamohseni)

## What to avoid

- `select *` in dashboard queries.
- Creating objects in `core`. You do not own `core`.
- Optimizing without runtime measurements.
- Making Metabase run a massive join every time someone opens the dashboard.

In [13]:
import os, re, time
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text

for path in ['sql', 'reports', 'screenshots']:
    Path(path).mkdir(exist_ok=True)

DB_HOST = os.getenv('QBC12_DB_HOST', '')    # this is in the excel file give in Quera
DB_PORT = os.getenv('QBC12_DB_PORT', '32112')
DB_NAME = os.getenv('QBC12_DB_NAME', 'qbc12_airbnb')
DB_USER = os.getenv('QBC12_DB_USER', '') or input('DB user: ').strip()
DB_PASSWORD = os.getenv('QBC12_DB_PASSWORD', '') or input('DB password: ').strip()
STUDENT_ID = os.getenv('QBC12_STUDENT_ID', '') or DB_USER.replace('student_', '')

safe_student = re.sub(r'[^a-zA-Z0-9_]', '_', STUDENT_ID.lower())
STUDENT_SCHEMA = f'{DB_USER}'
engine = create_engine(f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}', pool_pre_ping=True)
with engine.begin() as conn:
    conn.execute(text("SET statement_timeout = '30s'"))
    version = conn.execute(text('select version()')).scalar()
STUDENT_SCHEMA, version[:80]

('student_alireza_abouei',
 'PostgreSQL 16.14 (Debian 16.14-1.pgdg13+1) on x86_64-pc-linux-gnu, compiled by g')

## 1. Inspect before querying

You are not allowed to write the final query blind. Check columns and row counts first.

In [3]:
columns_sql = '''
select table_schema, table_name, column_name, data_type
from information_schema.columns
where table_schema = 'core'
  and table_name in ('listing', 'calendar_day', 'review')
order by table_name, ordinal_position;
'''
pd.read_sql(columns_sql, engine)

,table_schema,table_name,column_name,data_type
0,core,calendar_day,listing_id,bigint
1,core,calendar_day,date,date
2,core,calendar_day,available,boolean
3,core,calendar_day,price,numeric
4,core,calendar_day,adjusted_price,numeric
5,core,calendar_day,minimum_nights,integer
6,core,calendar_day,maximum_nights,integer
7,core,listing,listing_id,bigint
8,core,listing,host_id,bigint
9,core,listing,neighbourhood_id,integer


In [4]:
row_count_sql = '''
select 'core.listing' as table_name, count(*) as rows from core.listing
union all select 'core.calendar_day', count(*) from core.calendar_day
union all select 'core.review', count(*) from core.review;
'''
pd.read_sql(row_count_sql, engine)

,table_name,rows
0,core.listing,10480
1,core.calendar_day,3825200
2,core.review,501084


## 2. Create your sandbox schema

This is the only place you write database objects.

In [ ]:
# TODO 2.1
# Create your schema if it does not exist.
# Schema name is STUDENT_SCHEMA.

# Write your code here.


## 3. Build baseline SQL in pieces

Do not write one giant query first. Build the CTEs, test them, then combine.

In [5]:
# TODO 3.1
# Write calendar_30_sql.
# Required output: listing_id, avg_calendar_price_30, availability_30_rate.

calendar_30_sql = """
with max_calendar_date as (
    select max(date) as max_date
    from core.calendar_day
),
calendar_30 as (
    select
        c.listing_id,
        avg(coalesce(c.adjusted_price, c.price)) as avg_calendar_price_30,
        avg(case when c.available then 1.0 else 0.0 end) as availability_30_rate
    from core.calendar_day c
    cross join max_calendar_date m
    where c.date >= m.max_date - interval '29 days'
      and c.date <= m.max_date
    group by c.listing_id
)
select
    listing_id,
    avg_calendar_price_30,
    availability_30_rate
from calendar_30
order by listing_id
"""

In [6]:
# TODO 3.2
# Write review_counts_sql.
# Required output: listing_id, total_reviews.

review_counts_sql = """
select
    listing_id,
    count(*) as total_reviews
from core.review
group by listing_id
order by listing_id
"""

In [7]:
# TODO 3.3
# Combine the CTEs with core.listing into baseline_sql.
# Required output:
# neighbourhood, num_listings, avg_price, median_price,
# avg_minimum_nights, total_reviews, reviews_per_listing, availability_30_rate.

baseline_sql = '''
with max_calendar_date as (
    select max(date) as max_date
    from core.calendar_day
),
calendar_30 as (
    select
        c.listing_id,
        avg(coalesce(c.adjusted_price, c.price)) as avg_calendar_price_30,
        avg(case when c.available then 1.0 else 0.0 end) as availability_30_rate
    from core.calendar_day c
    cross join max_calendar_date m
    where c.date >= m.max_date - interval '29 days'
      and c.date <= m.max_date
    group by c.listing_id
),
review_counts as (
    select
        listing_id,
        count(*) as total_reviews
    from core.review
    group by listing_id
)
select
    l.neighbourhood_id::text as neighbourhood,
    count(*) as num_listings,
    round(avg(l.listing_price), 2) as avg_price,
    percentile_cont(0.5) within group (order by l.listing_price) as median_price,
    round(avg(l.minimum_nights), 2) as avg_minimum_nights,
    coalesce(sum(r.total_reviews), 0) as total_reviews,
    round(
        coalesce(sum(r.total_reviews), 0)::numeric / nullif(count(*), 0),
        2
    ) as reviews_per_listing,
    round(avg(c.availability_30_rate), 4) as availability_30_rate
from core.listing l
left join calendar_30 c
    on l.listing_id = c.listing_id
left join review_counts r
    on l.listing_id = r.listing_id
group by l.neighbourhood_id
order by num_listings desc
'''
Path('sql/01_baseline_neighbourhood_summary.sql').write_text(baseline_sql)

1321

In [8]:
def timed_read_sql(sql: str, repeats: int = 3):
    times = []
    last_df = None
    for _ in range(repeats):
        start = time.perf_counter()
        last_df = pd.read_sql(sql, engine)
        times.append(time.perf_counter() - start)
    return last_df, times

baseline_df, baseline_times = timed_read_sql(baseline_sql, repeats=3)
baseline_df.head(), baseline_times

(  neighbourhood  num_listings  avg_price  median_price  avg_minimum_nights  \
 0            22          1808     271.28         240.0                3.95   
 1             2          1207     315.88         245.5                4.01   
 2            20          1199     280.47         250.0                5.46   
 3             1           923     307.72         240.0                4.89   
 4             4           736     255.04         214.5                4.24   
 
    total_reviews  reviews_per_listing  availability_30_rate  
 0        62753.0                34.71                0.1988  
 1       106496.0                88.23                0.2859  
 2        45931.0                38.31                0.2321  
 3        76899.0                83.31                0.3140  
 4        26668.0                36.23                0.2091  ,
 [0.6216171890000624, 0.60024267100016, 0.6033131120002508])

## 4. Read the query plan

`EXPLAIN ANALYZE` actually runs the query. Look for big scans, expensive joins, and repeated work.

In [10]:
# TODO 4.1
# Run EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT) on baseline_sql.
# Save the plan to reports/baseline_explain_analyze.txt.

# Write your code here.
from pathlib import Path
from sqlalchemy import text

explain_sql = f"""
EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT)
{baseline_sql}
"""

with engine.begin() as conn:
    rows = conn.execute(text(explain_sql)).fetchall()

plan_text = "\n".join(row[0] for row in rows)

Path("reports/baseline_explain_analyze.txt").write_text(
    plan_text,
    encoding="utf-8",
)

print(plan_text[:5000])

Sort  (cost=59374.49..59374.54 rows=22 width=212) (actual time=349.881..350.014 rows=22 loops=1)
  Sort Key: (count(*)) DESC
  Sort Method: quicksort  Memory: 26kB
  Buffers: shared hit=13360 read=7736
  ->  GroupAggregate  (cost=59136.62..59374.00 rows=22 width=212) (actual time=343.939..349.984 rows=22 loops=1)
        Group Key: l.neighbourhood_id
        Buffers: shared hit=13360 read=7736
        ->  Sort  (cost=59136.62..59162.89 rows=10506 width=53) (actual time=343.302..344.303 rows=10480 loops=1)
              Sort Key: l.neighbourhood_id
              Sort Method: quicksort  Memory: 916kB
              Buffers: shared hit=13360 read=7736
              ->  Hash Left Join  (cost=58143.31..58434.88 rows=10506 width=53) (actual time=323.438..339.545 rows=10480 loops=1)
                    Hash Cond: (l.listing_id = r.listing_id)
                    Buffers: shared hit=13360 read=7736
                    ->  Hash Right Join  (cost=44416.38..44680.36 rows=10506 width=53) (actual ti

In [12]:
# TODO 4.2
# Write reports/explain_notes.md with 3 specific observations from the plan.
# Do not write vague nonsense like 'the query is slow'.

from pathlib import Path

Path("reports").mkdir(parents=True, exist_ok=True)

best_baseline = min(baseline_times)
avg_baseline = sum(baseline_times) / len(baseline_times)

explain_notes = f"""# EXPLAIN ANALYZE Notes

## Observation 1 — The final result is tiny, but the query does large intermediate work

The final query returns only 22 neighbourhood rows.

Evidence from the plan:

- Final `Sort` node: `actual time=349.881..350.014 rows=22`
- Final `GroupAggregate` node: `actual time=343.939..349.984 rows=22`
- Group key: `l.neighbourhood_id`

This means the output is small, but PostgreSQL still has to process much larger intermediate datasets before producing the final neighbourhood summary.

## Observation 2 — Calendar aggregation is the largest expensive part

The query filters the latest 30 days from `core.calendar_day` using the `idx_calendar_date` index.

Evidence from the plan:

- `Bitmap Index Scan on idx_calendar_date`
- `Bitmap Heap Scan on calendar_day c`
- Calendar rows processed: `rows=314400`
- Calendar heap blocks read: `Heap Blocks: exact=12416`
- Calendar scan time: `actual time=44.902..106.749`
- Calendar aggregation by listing: `HashAggregate`
- Calendar aggregation time: `actual time=234.317..243.188`
- Calendar aggregation memory: `Memory Usage: 4241kB`

So the index helps, but the query still reads and aggregates 314,400 calendar rows before joining to listings.

## Observation 3 — Review aggregation also adds noticeable work

The query aggregates `core.review` by `listing_id` before joining the result to listings.

Evidence from the plan:

- Review aggregation node: `Finalize HashAggregate`
- Group key: `review.listing_id`
- Review aggregate rows produced: `rows=9383`
- Review aggregation time: `actual time=73.248..74.929`
- Review buffers: `shared hit=480 read=7736`

This is smaller than the calendar work, but it is still repeated every time the baseline query runs.

## Baseline timing summary

Python-side timings from repeated runs:

- Run 1: `{baseline_times[0]:.4f}` seconds
- Run 2: `{baseline_times[1]:.4f}` seconds
- Run 3: `{baseline_times[2]:.4f}` seconds

Best Python-side runtime: `{best_baseline:.4f}` seconds
Average Python-side runtime: `{avg_baseline:.4f}` seconds

The database-side plan finishes around 350 ms, while the Python-side timing is around 0.60 seconds because it includes client overhead and loading the result into pandas.

## Optimization decision

The baseline query is correct, but it repeats calendar aggregation, review aggregation, joins, and final grouping every time it runs.

A materialized view is a good optimization because it stores the prepared neighbourhood-level result. Future reads can query the materialized view directly instead of recalculating the full baseline query.
"""

Path("reports/explain_notes.md").write_text(
    explain_notes,
    encoding="utf-8",
)

print(explain_notes)

# EXPLAIN ANALYZE Notes

## Observation 1 — The final result is tiny, but the query does large intermediate work

The final query returns only 22 neighbourhood rows.

Evidence from the plan:

- Final `Sort` node: `actual time=349.881..350.014 rows=22`
- Final `GroupAggregate` node: `actual time=343.939..349.984 rows=22`
- Group key: `l.neighbourhood_id`

This means the output is small, but PostgreSQL still has to process much larger intermediate datasets before producing the final neighbourhood summary.

## Observation 2 — Calendar aggregation is the largest expensive part

The query filters the latest 30 days from `core.calendar_day` using the `idx_calendar_date` index.

Evidence from the plan:

- `Bitmap Index Scan on idx_calendar_date`
- `Bitmap Heap Scan on calendar_day c`
- Calendar rows processed: `rows=314400`
- Calendar heap blocks read: `Heap Blocks: exact=12416`
- Calendar scan time: `actual time=44.902..106.749`
- Calendar aggregation by listing: `HashAggregate`
- Calendar a

## 5. Create a materialized view

Metabase should read from a prepared object, not a fresh monster join.

In [14]:
# TODO 5.1
# Create optimized_sql.
# It should create student_<you>.mv_airbnb_neighbourhood_summary and at least two indexes.

optimized_sql = f'''
drop materialized view if exists "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary;

create materialized view "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary as
with max_calendar_date as (
    select max(date) as max_date
    from core.calendar_day
),
calendar_30 as (
    select
        c.listing_id,
        avg(coalesce(c.adjusted_price, c.price)) as avg_calendar_price_30,
        avg(case when c.available then 1.0 else 0.0 end) as availability_30_rate
    from core.calendar_day c
    cross join max_calendar_date m
    where c.date >= m.max_date - interval '29 days'
      and c.date <= m.max_date
    group by c.listing_id
),
review_counts as (
    select
        listing_id,
        count(*) as total_reviews
    from core.review
    group by listing_id
)
select
    l.neighbourhood_id::text as neighbourhood,
    count(*) as num_listings,
    round(avg(l.listing_price), 2) as avg_price,
    percentile_cont(0.5) within group (order by l.listing_price) as median_price,
    round(avg(l.minimum_nights), 2) as avg_minimum_nights,
    coalesce(sum(r.total_reviews), 0) as total_reviews,
    round(
        coalesce(sum(r.total_reviews), 0)::numeric / nullif(count(*), 0),
        2
    ) as reviews_per_listing,
    round(avg(c.availability_30_rate), 4) as availability_30_rate
from core.listing l
left join calendar_30 c
    on l.listing_id = c.listing_id
left join review_counts r
    on l.listing_id = r.listing_id
group by l.neighbourhood_id;

create index idx_mv_airbnb_neighbourhood_summary_neighbourhood
on "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary (neighbourhood);

create index idx_mv_airbnb_neighbourhood_summary_num_listings
on "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary (num_listings desc);

create index idx_mv_airbnb_neighbourhood_summary_avg_price
on "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary (avg_price);
'''
Path('sql/02_create_materialized_view.sql').write_text(optimized_sql)

1890

In [21]:
# TODO 5.2
# Execute the materialized view SQL.

from sqlalchemy import text

with engine.begin() as conn:

    for statement in optimized_sql.split(";"):
        statement = statement.strip()
        if statement:
            conn.execute(text(statement))

print(f"Materialized view created in schema: {STUDENT_SCHEMA}")

Materialized view created in schema: student_alireza_abouei


In [17]:
check_sql = f'''select * from "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary order by num_listings desc limit 10;'''
pd.read_sql(check_sql, engine)

,neighbourhood,num_listings,avg_price,median_price,avg_minimum_nights,total_reviews,reviews_per_listing,availability_30_rate
0,22,1808,271.28,240.0,3.95,62753.0,34.71,0.1988
1,2,1207,315.88,245.5,4.01,106496.0,88.23,0.2859
2,20,1199,280.47,250.0,5.46,45931.0,38.31,0.2321
3,1,923,307.72,240.0,4.89,76899.0,83.31,0.3140
4,4,736,255.04,214.5,4.24,26668.0,36.23,0.2091
5,21,735,309.58,238.0,3.99,29444.0,40.06,0.2361
6,13,654,251.66,222.0,4.06,20448.0,31.27,0.1958
7,14,547,204.67,184.0,6.08,16461.0,30.09,0.1531
8,18,485,370.50,196.5,3.34,22623.0,46.65,0.1926
9,3,436,216.60,195.0,4.40,13988.0,32.08,0.1846


## 6. Compare latency

Numbers or it did not happen.

In [20]:
dashboard_sql = f'''
select neighbourhood, num_listings, avg_price, median_price,
       total_reviews, reviews_per_listing, availability_30_rate
from "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary
order by num_listings desc;
'''
dashboard_df, dashboard_times = timed_read_sql(dashboard_sql, repeats=5)
perf = pd.DataFrame([
    {'query': 'baseline_direct_query', 'best_seconds': min(baseline_times), 'avg_seconds': sum(baseline_times)/len(baseline_times)},
    {'query': 'materialized_view_read', 'best_seconds': min(dashboard_times), 'avg_seconds': sum(dashboard_times)/len(dashboard_times)},
])
perf['speedup_vs_baseline_best'] = perf.loc[0, 'best_seconds'] / perf['best_seconds']
perf

,query,best_seconds,avg_seconds,speedup_vs_baseline_best
0,baseline_direct_query,0.600243,0.608391,1.000000
1,materialized_view_read,0.311829,0.328334,1.924908


## 7. Metabase dashboard

Open the shared Metabase URL and create:

```text
QBC12 HW01 - <your-github-username> - Airbnb Ops
```

Required cards:

1. listings by neighbourhood
2. average price by neighbourhood
3. review activity by neighbourhood
4. availability rate by neighbourhood
5. top neighbourhoods table

Screenshot path:

```text
screenshots/metabase_dashboard.png
```

In [ ]:
# TODO 7.1
# Write reports/hw01_b_sql_performance.md.
# Include schema, runtimes, speedup, what changed, and Metabase screenshot/link.

# Write your code here.

In [ ]:
for file in ['sql/01_baseline_neighbourhood_summary.sql','sql/02_create_materialized_view.sql','reports/baseline_explain_analyze.txt','reports/explain_notes.md','reports/hw01_b_sql_performance.md']:
    assert Path(file).exists(), f'Missing {file}'
assert len(dashboard_df) > 0
perf

## Commit

```bash
git add sql reports screenshots notebooks
git commit -m "HW01-B SQL performance and Metabase dashboard"
```